### ⚠️ Patched for Local Execution
This notebook was originally designed for Google Colab. It has been automatically patched:
- Google Colab-specific imports and `drive.mount()` calls have been commented out
- Colab file paths (`/content/drive/...`) have been replaced with relative paths (`./`)
- `!pip install` commands have been commented out (install packages in your venv instead)

**To run locally:** activate your Python virtual environment first, then run this notebook in VS Code or Jupyter.

In [ ]:
# [PATCHED] !pip install transformers
# [PATCHED] !pip install accelerate -U

In [ ]:
# [PATCHED] from google.colab import drive
# [PATCHED] drive.mount('./')

In [ ]:
import zipfile
import pandas as pd
import os

working_folder='./ Drive/TransformersCode/02-ECommerce/CustomerFeedbackAnalysis/'

zip_file_path = working_folder+ 'ProductsReviews.zip'

save_output = working_folder + "roberta_base_fine_tuned"

with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extractall(working_folder)

csv_file_path = os.path.join(working_folder, 'ProductsReviews.csv')
df = pd.read_csv(csv_file_path)

df.head()

In [ ]:
df = df[["reviews.text","sentiment"]]

df.head()

In [ ]:
df.shape

In [ ]:
counts=df['sentiment'].value_counts()

counts

In [ ]:
plt = counts.sort_index().plot(kind='bar', title="Reviews by customers" )

plt.set_xlabel("Rating")
plt.set_ylabel("Count")
plt.plot()

In [ ]:
plt = counts.sort_index().plot(kind='pie', autopct='%1.1f%%', title="Reviews by customers")
plt.plot()

In [ ]:
def map_sentiment_to_class(sentiment):

    if sentiment=="Negative":
        return 0

    elif sentiment=="Neutral":
        return 1

    elif sentiment=="Positive":
        return 2

df.loc[:, "Class"] = df["sentiment"].apply(map_sentiment_to_class)

In [ ]:
df.head()

In [ ]:
total_rows = 300

class_rows=int(total_rows/3)

df_class_0 = df[df['Class'] == 0]
df_class_1 = df[df['Class'] == 1]
df_class_2 = df[df['Class'] == 2]


df_sample_0 = df_class_0.sample(n=min(class_rows, len(df_class_0)), random_state=42)
df_sample_1 = df_class_1.sample(n=min(class_rows, len(df_class_1)), random_state=42)
df_sample_2 = df_class_2.sample(n=min(class_rows, len(df_class_2)), random_state=42)


df_final = pd.concat([df_sample_0, df_sample_1, df_sample_2])

In [ ]:
counts=df_final['Class'].value_counts()

plt = counts.sort_index().plot(kind='pie', autopct='%1.1f%%', title="Reviews by customers")
plt.plot()

In [ ]:
from transformers import RobertaTokenizer, RobertaForSequenceClassification
import torch

tokenizer = RobertaTokenizer.from_pretrained('roberta-base')

model = RobertaForSequenceClassification.from_pretrained('roberta-base', num_labels=3)

In [ ]:
def get_accuracy_pt(model, tokenizer, df_param):

    df=df_param.copy()
    df["PredClass"] = 0

    for index, row in df.iterrows():

        sentence = row["reviews.text"]

        tokenized_input = tokenizer(sentence, return_tensors="pt", padding=True, truncation=True, max_length=512)

        output = model(**tokenized_input)

        logits = output.logits

        predicted_class = torch.argmax(logits).item()

        df.at[index, "PredClass"] = predicted_class

    correct_predictions = (df["Class"] == df["PredClass"]).sum()

    total_predictions = len(df)

    accuracy = correct_predictions / total_predictions
    return round(100*accuracy,2)

In [ ]:
from transformers import Trainer, TrainingArguments

from sklearn.metrics import accuracy_score

texts = df_final["reviews.text"].tolist()

labels = df_final["Class"].tolist()

from sklearn.model_selection import train_test_split

texts_train, texts_test, labels_train, labels_test = train_test_split(texts, labels, test_size=0.2, stratify=labels)

tokenized_texts_train = tokenizer(texts_train, return_tensors="pt", padding=True, truncation=True, max_length=512)

tokenized_texts_test = tokenizer(texts_test, return_tensors="pt", padding=True, truncation=True, max_length=512)


labels_train = torch.tensor(labels_train)
labels_test = torch.tensor(labels_test)


train_dataset = []
for i in range(len(tokenized_texts_train["input_ids"])):
    input_ids = tokenized_texts_train["input_ids"][i]
    attention_mask = tokenized_texts_train["attention_mask"][i]
    label = labels_train[i].item()
    train_dataset.append({'input_ids': input_ids, 'attention_mask': attention_mask, 'labels': label})


test_dataset = []
for i in range(len(tokenized_texts_test["input_ids"])):
    input_ids = tokenized_texts_test["input_ids"][i]
    attention_mask = tokenized_texts_test["attention_mask"][i]
    label = labels_test[i].item()
    test_dataset.append({'input_ids': input_ids, 'attention_mask': attention_mask, 'labels': label})


training_args = TrainingArguments(
    output_dir=save_output,
    num_train_epochs=8,
    per_device_train_batch_size=2,
    eval_strategy="epoch",
    report_to="none"
)

def compute_metrics(p):

    preds = p.predictions.argmax(-1)

    accuracy = accuracy_score(p.label_ids, preds)

    return {"accuracy": accuracy}

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)

trainer.train()

In [ ]:
train_results = trainer.evaluate(train_dataset)
train_accuracy = train_results["eval_accuracy"]

test_results = trainer.evaluate(test_dataset)
test_accuracy = test_results["eval_accuracy"]


print(f"Train Accuracy: {train_accuracy}")
print(f"Val Accuracy: {test_accuracy}")

In [ ]:
model.to("cpu")

all_accuracy = get_accuracy_pt(model, tokenizer ,df_final)
print(f"All Accuracy: {all_accuracy}")

In [ ]:
model.save_pretrained(save_output)

tokenizer.save_pretrained(save_output)